# Ground Motion Records Processing for Intensity Measure Extraction 

## Introduction

This Jupyter Notebook provides a workflow for reading ground motion record files and extracting distinct intensity measure (IM) levels across various IM types. Ground motion intensity measures are essential parameters that characterize the ground shaking and potential impact of seismic records on structures. By extracting these measures, we can better understand the demand imposed by ground motions on structural systems and eventually characterise a probabilistic demand-intensity model to building classes under investigation.

The main goals of this notebook are:

1. **Read and Process Ground Motion Records**: Load seismic record files. The files should be in a format that allows easy parsing and processing. In this example, each ground-motion record consists of an acceleration time-history and time-step files.

2. **Calculate the Response Spectrum**

3. **Extract Intensity Measure (IM) Levels**: For each ground motion record, calculate various intensity measures, such as:
   - Peak Ground Acceleration (PGA)
   - Peak Ground Velocity (PGV)
   - Peak Ground Displacement (PGD)
   - Spectral Acceleration (SA) at specified periods (e.g., 0.3s, 0.6s, 1.0s)
   - Average Spectral Acceleration (AvgSA) considering a range of 0.2T to 1.5T at user-specified periods (e.g., 0.3s, 0.6s, 1.0s)
   - Average Spectral Acceleration (AvgSA) considering a user-defined range of periods 
   - Arias Intensity (AI), Cumulative Absolute Velocity (CAV) and Significant Duration (D595)
   - Filtered Incremental Velocity 3 (FIV3)
  
Please note that the class used "IMCalculator" is always subject to updating to reflect state-of-the-art IM inclusion.

4. **Organize and Export Results**: Organize the extracted IM levels for each record, summarizing the results for analysis and potential use in structural modeling and assessment studies. The results are stored into a dictionary type variable and exported into a pickle file. 

This notebook provides a step-by-step guide to process ground motion records efficiently, extract a wide range of IM types, and present the data in an accessible format. Basic knowledge of earthquake engineering and data processing is recommended to fully utilize the material in this notebook.

---

## Import Libraries ## 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openquake.vmtk.imcalculator import imcalculator
from openquake.vmtk.utilities import sorted_alphanumeric, export_to_pkl

## Define Plotting Constants for OQ-VMTK Non-Native Plots ##

In [2]:
FONTSIZE_1 = 16
FONTSIZE_2 = 14
FONTSIZE_3 = 12

LINEWIDTH_1 = 3
LINEWIDTH_2 = 2
LINEWIDTH_3 = 1

RESOLUTION = 500

MARKER_SIZE_1 = 100
MARKER_SIZE_2 = 60
MARKER_SIZE_3 = 10

COLOR = "#399283"

## Load Acceleration Time-Histories ## 

The records used in this demonstration are derived from the datasets utilized in the development of the European Seismic Risk Model (2020). These records were previously post-processed in a separate demonstration file titled “IntensityMeasureProcessing.” 

The records are available on: https://gitlab.seismo.ethz.ch/efehr/esrm20_vulnerability/-/tree/master/scripts/vmtk/gmrs?ref_type=heads

If you use these records, please cite as follows:

X. Romão, N. Pereira, J.M. Castro, H. Crowley, V. Silva, L. Martins, & F. De Maio. (2021). European Building Vulnerability Data Repository. Zenodo. https://doi.org/10.5281/zenodo.4062410

In [ ]:
# Input the intensity measure types required for processing
IMT = [
    "PGA", "PGV", "PGD",
    "SA(0.3s)", "SA(0.6s)", "SA(1.0s)",
    "AvgSA", "AvgSA(0.3s)", "AvgSA(0.6s)", "AvgSA(1.0s)",
    "AI", "CAV", "D595",
]

# Load the records directory which contains two folders where the time
# and acceleration arrays of each ground-motion record are stored
gm_directory = "./in/records"

# Fetch the acceleration files
gmrs = sorted_alphanumeric(os.listdir(os.path.join(gm_directory, "acc")))

# Initialise the storage dictionary and name the "keys" in accordance
# with the defined IMT
imls = {}
for current_imt in IMT:
    imls[current_imt] = []

## Process Intensity Measures from Ground-Motion Records ## 

The `imcalculator` class computes a wide range of intensity measures (IMs) from an earthquake acceleration time-history. Examples of these IMs are explained below:


* **Spectral Acceleration (SA):**

$$
SA(T) = \max |\ddot{u}(t)|
$$

where $\ddot{u}(t)$ is the acceleration response of a single-degree-of-freedom (SDOF) system with natural period $T$ and damping ratio $\xi$, calculated via Newmark-beta integration.


* **Spectral Velocity (SV) and Displacement (SD):**

$$
SV(T) = \max |\dot{u}(t)|, \quad SD(T) = \max |u(t)|
$$

* **Average Spectral Acceleration (AvgSA)** over a range of periods $T_{i}$:

$$
AvgSA = \left( \prod_{i=1}^{N} SA(T_i) \right)^{\frac{1}{N}}
$$

where $T_{i}$ are periods in a range around the target period (e.g., $T_{i}$ in [0.2 T, 1.5 T] or a user-defined set of periods.

* **Peak Ground Acceleration (PGA):**

$$
PGA = \max |a(t)|
$$

* **Peak Ground Velocity (PGV):**

$$
PGV = \max |v(t)|, \quad v(t) = \int_0^T a(\tau) , d\tau
$$

* **Peak Ground Displacement (PGD):**

$$
PGD = \max |d(t)|, \quad d(t) = \int_0^T v(\tau) , d\tau
$$


* **Arias Intensity (AI):**

$$
AI = \frac{\pi}{2 g} \int_0^T [a(t)]^2 , dt
$$

* **Cumulative Absolute Velocity (CAV):**

$$
CAV = \int_0^T |a(t)| , dt
$$


* **Significant Duration (D5%-95%)**: time interval containing 5% to 95% of the total Arias intensity:

$$
D_{595} = t_{95\%} - t_{5\%}, \quad \text{where } t_p \text{ satisfies } \frac{AI(t_p)}{AI_\mathrm{total}} = p
$$


In [ ]:
# Loop over the files
for i in range(len(gmrs)):

    # Load the acceleration time-histories and the time-step files
    current_acc = pd.read_csv(
        os.path.join(gm_directory, "acc", f"acc_{i}.csv"), header=None
    ).to_numpy().flatten()
    current_dts = pd.read_csv(
        os.path.join(gm_directory, "dts", f"dts_{i}.csv"), header=None
    ).to_numpy().flatten()

    # Define the time-step of the acceleration time-history
    dt_gm = (current_dts[1] - current_dts[0]).item()

    # Plot the time history only for the first ground-motion record
    if i == 0:
        plt.plot(current_dts, current_acc, color=COLOR, lw=LINEWIDTH_3)
        plt.xlabel("Pseudo-time [s]", fontsize=FONTSIZE_1)
        plt.ylabel("Acceleration [g]", fontsize=FONTSIZE_1)
        plt.grid(visible=True, which="major")
        plt.grid(visible=True, which="minor")
        plt.xlim([0.00, np.max(current_dts)])
        plt.show()

    # Initialise the Intensity Measure Calculator class
    im_calc = imcalculator(current_acc, dt_gm)

    # Calculate amplitude-based intensity measures: PGA, PGV, PGD
    pga, pgv, pgd = im_calc.get_amplitude_ims()

    # Calculate duration-based intensity measures individually
    ai   = im_calc.get_arias_intensity()
    cav  = im_calc.get_cav()
    t595 = im_calc.get_significant_duration()

    # Calculate and plot the acceleration, velocity and displacement spectra
    periods, sd, sv, sa = im_calc.get_spectrum()

    # Plot the spectra only for the first ground-motion record
    if i == 0:

        # Acceleration Spectrum
        plt.plot(periods, sa, color=COLOR, lw=LINEWIDTH_3)
        plt.xlabel("Period [s]", fontsize=FONTSIZE_1)
        plt.ylabel("Pseudo Spectral Acceleration [g]", fontsize=FONTSIZE_1)
        plt.grid(visible=True, which="major")
        plt.grid(visible=True, which="minor")
        plt.xlim([0.00, np.max(periods)])
        plt.show()

        # Velocity Spectrum
        plt.plot(periods, sv, color=COLOR, lw=LINEWIDTH_3)
        plt.xlabel("Period [s]", fontsize=FONTSIZE_1)
        plt.ylabel("Pseudo Spectral Velocity [m/s]", fontsize=FONTSIZE_1)
        plt.grid(visible=True, which="major")
        plt.grid(visible=True, which="minor")
        plt.xlim([0.00, np.max(periods)])
        plt.show()

        # Displacement Spectrum
        plt.plot(periods, sd, color=COLOR, lw=LINEWIDTH_3)
        plt.xlabel("Period [s]", fontsize=FONTSIZE_1)
        plt.ylabel("Pseudo Spectral Displacement [m]", fontsize=FONTSIZE_1)
        plt.grid(visible=True, which="major")
        plt.grid(visible=True, which="minor")
        plt.xlim([0.00, np.max(periods)])
        plt.show()

    # Calculate the spectral acceleration values at distinct periods
    sa03 = im_calc.get_sa(0.3)
    sa06 = im_calc.get_sa(0.6)
    sa10 = im_calc.get_sa(1.0)

    # Calculate the average spectral acceleration values at distinct periods
    # with an auto-defined range of 0.2T to 1.5T
    avgsa03 = im_calc.get_saavg(0.3)
    avgsa06 = im_calc.get_saavg(0.6)
    avgsa10 = im_calc.get_saavg(1.0)

    # Calculate the average spectral acceleration with a user-defined range
    periods_list = np.linspace(0.1, 1.0, 10)
    avgsa = im_calc.get_saavg_user_defined(periods_list)

    # Print outputs only for first record
    if i == 0:
        print(f"Peak Ground Acceleration (PGA): {pga:.4f} g")
        print(f"Peak Ground Velocity (PGV): {pgv:.4f} m/s")
        print(f"Peak Ground Displacement (PGD): {pgd:.4f} m")

        print(f"Arias Intensity (AI): {ai:.4f} m/s")
        print(f"Cumulative Absolute Velocity (CAV): {cav:.4f} m/s")
        print(f"Significant Duration (5%-95% AI): {t595:.4f} s")

        print(f"Spectral Acceleration (SA at 0.3s): {sa03:.4f} g")
        print(f"Spectral Acceleration (SA at 0.6s): {sa06:.4f} g")
        print(f"Spectral Acceleration (SA at 1.0s): {sa10:.4f} g")

        print(f"Average Spectral Acceleration (AvgSA at 0.3s): {avgsa03:.4f} g")
        print(f"Average Spectral Acceleration (AvgSA at 0.6s): {avgsa06:.4f} g")
        print(f"Average Spectral Acceleration (AvgSA at 1.0s): {avgsa10:.4f} g")
        print(f"Average Spectral Acceleration (AvgSA at [0.1-1.0]): {avgsa:.4f} g")

    # Store the intensity measures in the dictionary
    for current_imt in IMT:
        if current_imt == "PGA":
            imls[current_imt].append(pga)
        elif current_imt == "PGV":
            imls[current_imt].append(pgv)
        elif current_imt == "PGD":
            imls[current_imt].append(pgd)
        elif current_imt == "SA(0.3s)":
            imls[current_imt].append(sa03)
        elif current_imt == "SA(0.6s)":
            imls[current_imt].append(sa06)
        elif current_imt == "SA(1.0s)":
            imls[current_imt].append(sa10)
        elif current_imt == "AvgSA":
            imls[current_imt].append(avgsa)
        elif current_imt == "AvgSA(0.3s)":
            imls[current_imt].append(avgsa03)
        elif current_imt == "AvgSA(0.6s)":
            imls[current_imt].append(avgsa06)
        elif current_imt == "AvgSA(1.0s)":
            imls[current_imt].append(avgsa10)
        elif current_imt == "AI":
            imls[current_imt].append(ai)
        elif current_imt == "CAV":
            imls[current_imt].append(cav)
        elif current_imt == "D595":
            imls[current_imt].append(t595)

# Export to pickle format
export_to_pkl(os.path.join("out", "imls_esrm20.pkl"), imls)